In [1]:
import pandas as pd
import os
import warnings

warnings.filterwarnings('ignore')

In [2]:
# df=pd.read_csv('stlouis2025obweekday_export_odbc.csv')
df=pd.read_csv('Transit DB Data.csv')
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_DATE_,ELVIS_USER_CHANGE_7_C_FIELD_,ELVIS_USER_CHANGE_7_C_REASON_,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,19567,NaN,-1,en,2025-04-04 15:01:35,2025-04-04 15:01:35,44.225.68.189,NaN,-18000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19568,NaN,-1,en,2025-04-04 15:09:31,2025-04-04 15:09:31,68.0.158.46,NaN,-18000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19569,NaN,-1,bs,2025-04-04 18:26:46,2025-04-04 18:26:46,68.0.158.46,NaN,-18000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,19570,NaN,-1,zh-Hans,2025-04-04 19:15:01,2025-04-04 19:15:01,68.0.158.46,NaN,-18000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19571,NaN,-1,zh-Hans,2025-04-04 19:25:08,2025-04-04 19:25:08,68.0.158.46,NaN,-18000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# df1 = pd.read_csv('etc_research_8383_export_odbc.csv', header=0, skiprows=[1])
df1 = pd.read_csv('Community DB Data.csv')

In [4]:
df1.head()

,id,submitdate,lastpage,startlanguage,seed,token,startdate,datestamp,ipaddr,refurl,...,Qxxyy_03,mtoken,Extra,Validation_01,Validation_02,Validation_03,Incentive_01,Incentive_02,Incentive_03,END
0,94,NaN,4,en,1499173462,NaN,2025-03-29 00:19:11,2025-03-29 00:19:11,43.134.141.244,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,95,2025-03-29 09:58:12,8,en,1162417561,NaN,2025-03-29 09:57:33,2025-03-29 09:58:12,68.0.158.46,https://stl-od.etc-research.com/,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,96,NaN,4,en,707014212,NaN,2025-03-29 10:18:06,2025-03-29 10:18:06,44.244.86.32,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,97,NaN,4,en,1909331675,NaN,2025-03-31 09:34:31,2025-03-31 09:34:31,52.112.49.156,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,98,NaN,4,en,2069008216,NaN,2025-03-31 12:05:23,2025-03-31 12:05:23,54.185.36.53,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def check_all_characters_present(df, columns_to_check):
    """
    Returns the list of DataFrame columns that match any cleaned version
    of the columns in `columns_to_check`, using a custom cleaning rule.
    """
    # Define a helper function to clean column names
    def clean(s):
        return ''.join(c for c in s.lower() if c not in {'_', '[', ']', ' ', '#'})

    # Create a set of cleaned column names to check for faster lookup
    columns_to_check_cleaned = set(map(clean, columns_to_check))

    # Filter df columns that match the cleaned names
    matching_columns = [col for col in df.columns if clean(col) in columns_to_check_cleaned]

    return matching_columns

In [42]:
phone_column_check=['followupsmsphone']
phone_name_column_check=['followupsmsname']
phone_column=check_all_characters_present(df,phone_column_check)
phone_name_column=check_all_characters_present(df,phone_name_column_check)
phone_column,phone_name_column

(['FOLLOWUP_SMS_PHONE_'], ['FOLLOWUP_SMS_NAME_'])

In [7]:
df[df[phone_column[0]].notna()][['id',phone_column[0]]]

,id,FOLLOWUP_SMS_PHONE_
20,19587,3.147121e+09
27,19596,6.187671e+09
28,19597,5.734349e+09
38,19608,7.302519e+09
43,19618,8.153366e+09
61,19659,6.189542e+09
75,19724,6.183436e+09
78,19739,2.488757e+09
79,19740,6.183041e+09
80,19741,3.149411e+09


In [8]:
def format_us_phone_numbers(df, phone_column):
    """
    Adds '+1' prefix to US phone numbers in the specified phone column,
    only if the number is not already in international format.
    """
    def format_number(num):
        if pd.isna(num):
            return num
        # Convert to string if it's a float
        num_str = str(num).strip()
        
        # If it's already in the format with a plus sign, just return it
        if num_str.startswith('+'):
            return num_str
        
        # Remove any non-numeric characters (like spaces or dashes) before adding '+1'
        num_str = ''.join(filter(str.isdigit, num_str))
        
        # Add the country code if it's not already present
        return '+1' + num_str

    df[phone_column[0]] = df[phone_column[0]].apply(format_number)
    return df


In [9]:
df= format_us_phone_numbers(df, phone_column)

In [10]:
df[df[phone_column[0]].notna()][['id',phone_column[0]]]

,id,FOLLOWUP_SMS_PHONE_
20,19587,+131471205250
27,19596,+161876713990
28,19597,+157343492140
38,19608,+173025186500
43,19618,+181533655110
61,19659,+161895424830
75,19724,+161834358110
78,19739,+124887574160
79,19740,+161830409460
80,19741,+131494105930


In [11]:
df[phone_column[0]]

0               NaN
1               NaN
2               NaN
3               NaN
4               NaN
          ...      
91    +131466634000
92              NaN
93    +161872284300
94    +161886635750
95              NaN
Name: FOLLOWUP_SMS_PHONE_, Length: 96, dtype: object

In [12]:
mainsid_column_check=['mainsid']
mainsid_column=check_all_characters_present(df1,mainsid_column_check)
mainsid_column

['MAINSID']

In [13]:
response_column_check=['responseid']
response_column=check_all_characters_present(df1,response_column_check)
response_column

['RESPONSEID']

In [14]:
new_df=df[df[phone_column[0]].notna()][['id',phone_column[0]]]

In [15]:
new_df[response_column[0]] = new_df['id'].astype(str)

In [16]:
new_df.columns

Index(['id', 'FOLLOWUP_SMS_PHONE_', 'RESPONSEID'], dtype='object')

In [17]:
# new_df1=df1[df1[response_column[0]]=='19587']
new_df1=df1[df1[response_column[0]]=='19471']

In [18]:
df1[response_column].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   RESPONSEID  90 non-null     object
dtypes: object(1)
memory usage: 992.0+ bytes


In [19]:
print(list(df1.columns))

['id', 'submitdate', 'lastpage', 'startlanguage', 'seed', 'token', 'startdate', 'datestamp', 'ipaddr', 'refurl', 'SURVEYINFO_ComName', 'SURVEYINFO_ST', 'SURVEYINFO_YR', 'MAINSID', 'RESPONSEID', 'switchExtension', 'SURVEYNAME', 'switch2', 'user', 'SURVEYNUMBER', 'idbkp', 'IPGL_IPaddress', 'IPGL_country', 'IPGL_region', 'IPGL_city', 'IPGL_latitude', 'IPGL_longitude', 'IPGL_zip', 'IPGL_timeZone', 'Q1_01', 'Q1_02', 'Q1_03', 'Q1_04', 'Q1_05', 'Q2_01', 'Q2_02', 'Q2_03', 'Q2_04', 'Q2_05', 'Q2_06', 'Q2_07', 'Q3', 'Q3a_01', 'Q3a_02', 'Q3a_03', 'Q3a_04', 'Q3a_05', 'Q3a_06', 'Q3a_07', 'Q4', 'Q4a_01', 'Q4a_02', 'Q4a_03', 'Q4a_04', 'Q4a_05', 'Q4a_06', 'Q4a_07', 'Q5_01', 'Q5_02', 'Q5_03', 'Q6', 'Q6a_01', 'Q6a_02', 'Q6a_03', 'Q6a_04', 'Q6a_05', 'Q6a_06', 'Q6a_07', 'Q6a7', 'Q7', 'Q8_01', 'Q8_02', 'Q8_03', 'Q8_04', 'Q8_05', 'Q8_06', 'Q8_07', 'Q8_08', 'Q8_09', 'Q8_10', 'Q9_01', 'Q9_02', 'Q9x15', 'Q10_01', 'Q10_02', 'Q10x8', 'Q11', 'Qx', 'Qxy_C2NAME', 'Qxy_C2TEXT', 'Qxy_C2EMAIL', 'Qxy_C2ZIP', 'Qxy_C2ADDR

In [20]:
new_df1[[response_column[0],'Q1_01', 'Q1_02', 'Q1_03', 'Q1_04', 'Q1_05', 'Q2_01', 'Q2_02', 'Q2_03', 'Q2_04', 'Q2_05', 'Q2_06', 'Q2_07', 'Q3', 'Q3a_01', 'Q3a_02', 'Q3a_03', 'Q3a_04', 'Q3a_05', 'Q3a_06', 'Q3a_07', 'Q4', 'Q4a_01', 'Q4a_02', 'Q4a_03', 'Q4a_04', 'Q4a_05', 'Q4a_06', 'Q4a_07', 'Q5_01', 'Q5_02', 'Q5_03', 'Q6', 'Q6a_01', 'Q6a_02', 'Q6a_03', 'Q6a_04', 'Q6a_05', 'Q6a_06', 'Q6a_07', 'Q6a7', 'Q7', 'Q8_01', 'Q8_02', 'Q8_03', 'Q8_04', 'Q8_05', 'Q8_06', 'Q8_07', 'Q8_08', 'Q8_09', 'Q8_10', 'Q9_01', 'Q9_02', 'Q9x15', 'Q10_01', 'Q10_02', 'Q10x8', 'Q11']]

,RESPONSEID,Q1_01,Q1_02,Q1_03,Q1_04,Q1_05,Q2_01,Q2_02,Q2_03,Q2_04,...,Q8_08,Q8_09,Q8_10,Q9_01,Q9_02,Q9x15,Q10_01,Q10_02,Q10x8,Q11
45,19471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,19471,Somewhat Comfortable,NaN,NaN,NaN,Somewhat Uncomfortable,No,No,Yes,No,...,Fair,Good,Good,01. More frequent rush-hour service,10. Better lighting/seating/shelters at stops,NaN,2. Earlier or later service hours,7. Easier access to transit information,NaN,I spend more than 2 hours a day commuting on t...
82,19471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# new_df1[[
#     'Q1_01', 'Q1_02', 'Q1_03', 'Q1_04', 'Q1_05', 'Q2_01', 'Q2_02', 'Q2_03', 'Q2_04', 'Q2_05', 'Q2_06', 'Q2_07',
#     'Q3', 'Q3a_01', 'Q3a_02', 'Q3a_03', 'Q3a_04', 'Q3a_05', 'Q3a_06', 'Q3a_07',
#     'Q4', 'Q4a_01', 'Q4a_02', 'Q4a_03', 'Q4a_04', 'Q4a_05', 'Q4a_06', 'Q4a_07',
#     'Q5_01', 'Q5_02', 'Q5_03', 'Q6', 'Q6a_01', 'Q6a_02', 'Q6a_03', 'Q6a_04', 'Q6a_05', 'Q6a_06', 'Q6a_07', 'Q6a7',
#     'Q7', 'Q8_01', 'Q8_02', 'Q8_03', 'Q8_04', 'Q8_05', 'Q8_06', 'Q8_07', 'Q8_08', 'Q8_09', 'Q8_10',
#     'Q9_01', 'Q9_02', 'Q9x15', 'Q10_01', 'Q10_02', 'Q10x8', 'Q11'
# ]].notna().sum()

In [22]:
# new_df1[[
#     'Q1_01', 'Q1_02', 'Q1_03', 'Q1_04', 'Q1_05', 'Q2_01', 'Q2_02', 'Q2_03', 'Q2_04', 'Q2_05', 'Q2_06', 'Q2_07',
#     'Q3', 'Q3a_01', 'Q3a_02', 'Q3a_03', 'Q3a_04', 'Q3a_05', 'Q3a_06', 'Q3a_07',
#     'Q4', 'Q4a_01', 'Q4a_02', 'Q4a_03', 'Q4a_04', 'Q4a_05', 'Q4a_06', 'Q4a_07',
#     'Q5_01', 'Q5_02', 'Q5_03', 'Q6', 'Q6a_01', 'Q6a_02', 'Q6a_03', 'Q6a_04', 'Q6a_05', 'Q6a_06', 'Q6a_07', 'Q6a7',
#     'Q7', 'Q8_01', 'Q8_02', 'Q8_03', 'Q8_04', 'Q8_05', 'Q8_06', 'Q8_07', 'Q8_08', 'Q8_09', 'Q8_10',
#     'Q9_01', 'Q9_02', 'Q9x15', 'Q10_01', 'Q10_02', 'Q10x8', 'Q11'
# ]].isna().all()

In [23]:
merged_df = pd.merge(new_df, df1, left_on=response_column[0], right_on=response_column[0], how='left')
# merged_df.drop_duplicates(subset=response_column[0],inplace=True)

In [24]:
merged_df.drop_duplicates(subset=response_column[0])

,id_x,FOLLOWUP_SMS_PHONE_,RESPONSEID,id_y,submitdate,lastpage,startlanguage,seed,token,startdate,...,Qxxyy_03,mtoken,Extra,Validation_01,Validation_02,Validation_03,Incentive_01,Incentive_02,Incentive_03,END
0,19587,+131471205250,19587,181.0,NaN,4.0,en,1.987852e+09,NaN,2025-04-07 11:14:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19596,+161876713990,19596,182.0,NaN,4.0,en,2.065389e+09,NaN,2025-04-07 12:01:43,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19597,+157343492140,19597,183.0,NaN,4.0,en,1.050353e+09,NaN,2025-04-07 12:08:08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,19608,+173025186500,19608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,19618,+181533655110,19618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,19659,+161895424830,19659,193.0,2025-04-07 15:26:17,8.0,en,1.522323e+09,NaN,2025-04-07 15:22:09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,19724,+161834358110,19724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,19739,+124887574160,19739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,19740,+161830409460,19740,197.0,2025-04-07 18:41:36,8.0,en,1.399710e+09,NaN,2025-04-07 18:31:25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,19741,+131494105930,19741,198.0,2025-04-07 18:41:27,8.0,en,6.784551e+08,NaN,2025-04-07 18:38:42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
def count_non_null(row):
    return row.notna().sum()

In [26]:
merged_df['non_null_count'] = merged_df.apply(count_non_null, axis=1)
merged_df=merged_df.sort_values('non_null_count', ascending=False).drop_duplicates(subset=[response_column[0]], keep='first').drop(columns=['non_null_count'])

In [27]:
# asasasasasasasas

In [28]:
merged_df

,id_x,FOLLOWUP_SMS_PHONE_,RESPONSEID,id_y,submitdate,lastpage,startlanguage,seed,token,startdate,...,Qxxyy_03,mtoken,Extra,Validation_01,Validation_02,Validation_03,Incentive_01,Incentive_02,Incentive_03,END
16,19767,+161886635750,19767,203.0,2025-04-07 21:28:55,8.0,en,2.089309e+09,NaN,2025-04-07 21:21:07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,19751,+161852024010,19751,199.0,2025-04-07 20:12:36,8.0,en,7.096116e+08,NaN,2025-04-07 20:10:21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,19741,+131494105930,19741,198.0,2025-04-07 18:41:27,8.0,en,6.784551e+08,NaN,2025-04-07 18:38:42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19587,+131471205250,19587,214.0,2025-04-08 10:52:26,8.0,en,1.078485e+09,NaN,2025-04-08 10:47:18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,19659,+161895424830,19659,193.0,2025-04-07 15:26:17,8.0,en,1.522323e+09,NaN,2025-04-07 15:22:09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,19740,+161830409460,19740,197.0,2025-04-07 18:41:36,8.0,en,1.399710e+09,NaN,2025-04-07 18:31:25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,19596,+161876713990,19596,190.0,2025-04-07 13:15:12,8.0,en,1.520751e+08,NaN,2025-04-07 13:08:06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19597,+157343492140,19597,183.0,NaN,4.0,en,1.050353e+09,NaN,2025-04-07 12:08:08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,19618,+181533655110,19618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,19739,+124887574160,19739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
for _, row in merged_df.iterrows():
    response_id = row[response_column[0]]
    mainsid = row[mainsid_column[0]]
    url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
    print(url)


https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19767
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19751
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19741
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19587
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19659
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19740
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19596
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19597
https://etc-research.com/index.php/8383?lang=en&MAINSID=nan&RESPONSEID=19618
https://etc-research.com/index.php/8383?lang=en&MAINSID=nan&RESPONSEID=19739
https://etc-research.com/index.php/8383?lang=en&MAINSID=nan&RESPONSEID=19608
https://etc-research.com/index.php/8383?lang=en&MAINSID=nan&RESPONSEID=19752
https://etc-research.com/index.php/8383?lang=en&MAIN

In [30]:
print(list(merged_df.columns))

['id_x', 'FOLLOWUP_SMS_PHONE_', 'RESPONSEID', 'id_y', 'submitdate', 'lastpage', 'startlanguage', 'seed', 'token', 'startdate', 'datestamp', 'ipaddr', 'refurl', 'SURVEYINFO_ComName', 'SURVEYINFO_ST', 'SURVEYINFO_YR', 'MAINSID', 'switchExtension', 'SURVEYNAME', 'switch2', 'user', 'SURVEYNUMBER', 'idbkp', 'IPGL_IPaddress', 'IPGL_country', 'IPGL_region', 'IPGL_city', 'IPGL_latitude', 'IPGL_longitude', 'IPGL_zip', 'IPGL_timeZone', 'Q1_01', 'Q1_02', 'Q1_03', 'Q1_04', 'Q1_05', 'Q2_01', 'Q2_02', 'Q2_03', 'Q2_04', 'Q2_05', 'Q2_06', 'Q2_07', 'Q3', 'Q3a_01', 'Q3a_02', 'Q3a_03', 'Q3a_04', 'Q3a_05', 'Q3a_06', 'Q3a_07', 'Q4', 'Q4a_01', 'Q4a_02', 'Q4a_03', 'Q4a_04', 'Q4a_05', 'Q4a_06', 'Q4a_07', 'Q5_01', 'Q5_02', 'Q5_03', 'Q6', 'Q6a_01', 'Q6a_02', 'Q6a_03', 'Q6a_04', 'Q6a_05', 'Q6a_06', 'Q6a_07', 'Q6a7', 'Q7', 'Q8_01', 'Q8_02', 'Q8_03', 'Q8_04', 'Q8_05', 'Q8_06', 'Q8_07', 'Q8_08', 'Q8_09', 'Q8_10', 'Q9_01', 'Q9_02', 'Q9x15', 'Q10_01', 'Q10_02', 'Q10x8', 'Q11', 'Qx', 'Qxy_C2NAME', 'Qxy_C2TEXT', 'Qxy_C

In [31]:
survey_end_columns_check=['incentive01','incentive02','incentive03','end']
survey_end_columns=check_all_characters_present(merged_df,survey_end_columns_check)
survey_end_columns.sort()
survey_end_columns

['END', 'Incentive_01', 'Incentive_02', 'Incentive_03']

In [32]:
survey_question_columns_check = [
    'q101', 'q102', 'q103', 'q104', 'q105', 'q201', 'q202', 'q203', 'q204', 'q205', 'q206', 'q207',
    'q3', 'q3a01', 'q3a02', 'q3a03', 'q3a04', 'q3a05', 'q3a06', 'q3a07',
    'q4', 'q4a01', 'q4a02', 'q4a03', 'q4a04', 'q4a05', 'q4a06', 'q4a07',
    'q501', 'q502', 'q503', 'q6', 'q6a01', 'q6a02', 'q6a03', 'q6a04', 'q6a05', 'q6a06', 'q6a07', 'q6a7',
    'q7', 'q801', 'q802', 'q803', 'q804', 'q805', 'q806', 'q807', 'q808', 'q809', 'q810',
    'q901', 'q902', 'q9x15', 'q1001', 'q1002', 'q10x8', 'q11'
]
survey_question_columns = check_all_characters_present(merged_df, survey_question_columns_check)
survey_question_columns.sort()

In [33]:
print(survey_question_columns)

['Q10_01', 'Q10_02', 'Q10x8', 'Q11', 'Q1_01', 'Q1_02', 'Q1_03', 'Q1_04', 'Q1_05', 'Q2_01', 'Q2_02', 'Q2_03', 'Q2_04', 'Q2_05', 'Q2_06', 'Q2_07', 'Q3', 'Q3a_01', 'Q3a_02', 'Q3a_03', 'Q3a_04', 'Q3a_05', 'Q3a_06', 'Q3a_07', 'Q4', 'Q4a_01', 'Q4a_02', 'Q4a_03', 'Q4a_04', 'Q4a_05', 'Q4a_06', 'Q4a_07', 'Q5_01', 'Q5_02', 'Q5_03', 'Q6', 'Q6a7', 'Q6a_01', 'Q6a_02', 'Q6a_03', 'Q6a_04', 'Q6a_05', 'Q6a_06', 'Q6a_07', 'Q7', 'Q8_01', 'Q8_02', 'Q8_03', 'Q8_04', 'Q8_05', 'Q8_06', 'Q8_07', 'Q8_08', 'Q8_09', 'Q8_10', 'Q9_01', 'Q9_02', 'Q9x15']


In [34]:
# data_to_save = []


In [35]:
for _, row in merged_df.iterrows():
    mainsid = row[mainsid_column[0]]
    
    # Check if mainsid is not missing
    if pd.notna(mainsid):
        response_id = row[response_column[0]]
        phone_number = row[phone_column[0]]
        counter=1
        url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
        print(url)

#         # Append data to the list
#         data_to_save.append([phone_number, response_id, mainsid, url])

https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19767
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19751
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19741
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19587
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19659
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19740
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19596
https://etc-research.com/index.php/8383?lang=en&MAINSID=932964&RESPONSEID=19597


In [36]:
# data_to_save = []


# for _, row in merged_df.iterrows():
#     response_id = str(row[response_column[0]])
#     phone_number = row[phone_column[0]]
#     mainsid = row[mainsid_column[0]]
#     count
#     # Skip if mainsid is missing or empty
#     if pd.isna(mainsid) or str(mainsid).strip() == "":
#         continue

#     # 1. Survey completed?
#     if any(pd.notna(row[col]) for col in survey_end_columns):
#         status = "Completed"
#         continue

#     # 2. Partial survey?
#     elif any(pd.notna(row[col]) for col in survey_question_columns):
#         # Send RESUME LINK
#         resume_url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
#         message_text = f"Please complete this short follow-up survey: {resume_url}"
#         status = "Partial - Resume Sent"
#         count+=1

#     # 3. Never started
#     else:
#         # Send full survey link
#         full_url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
#         message_text = f"Reminder to answer this short follow-up survey: {full_url}"
#         status = "Not Started - Survey Link Sent"
#         count+=1
#     print(message_text)

#     # Send SMS
# #     sms_message = SmsMessage(source="python", body=message_text, to=str(phone_number))
# #     sms_messages = clicksend_client.SmsMessageCollection(messages=[sms_message])

# #     try:
# #         api_response = api_instance.sms_send_post(sms_messages)
# #         print(f"Sent to {phone_number}: {api_response}")
# #     except ApiException as e:
# #         print(f"Failed to send to {phone_number}: {e}")
# #         status = "Failed"

#     # Log info
#     data_to_save.append([phone_number, response_id, mainsid, message_text, status])


In [37]:
# csv_filename = "survey_links_sent.csv"

# # Create a DataFrame from the collected data
# df_to_save = pd.DataFrame(data_to_save, columns=["Phone Number", "Response ID", "Main SID", "URL",'status'])

# # Append to the CSV file (or create it if it doesn't exist)
# df_to_save.to_csv(csv_filename, mode='w',index=False)

In [41]:
data_to_save = []
csv_filename = "survey_links_sent.csv"

# Read existing data from the CSV if it exists
if os.path.exists(csv_filename):
    df_existing = pd.read_csv(csv_filename)
else:
    # Create an empty DataFrame if the file does not exist
    df_existing = pd.DataFrame(columns=["Phone Number", "Response ID", "Main SID", "URL", "Status", "Count"])

# Iterate through the merged_df to process both existing and new records
for _, row in merged_df.iterrows():
    response_id = int(row[response_column[0]])
    print(response_id,type(response_id))
    phone_number = row[phone_column[0]]
    mainsid = row[mainsid_column[0]]
    print(mainsid,type(mainsid))

    # Skip if mainsid is missing or empty
    if pd.isna(mainsid) or str(mainsid).strip() == "":
        continue

    # Check if the response_id already exists in the CSV
    matching_row = df_existing[df_existing["Response ID"] == response_id]
#     print(matching_row)
    # If the response_id is found, process the existing record
    if not matching_row.empty:
        print('inside matching data')
        count = int(matching_row["Count"].iloc[0])  # Get the current count from the file
        
        # If the count is <= 3, proceed with the message sending logic
        if count <=2:
            # 1. Survey completed?
            if any(pd.notna(row[col]) for col in survey_end_columns):
                status = "Completed"
                message_text = ""
            # 2. Partial survey?
            elif any(pd.notna(row[col]) for col in survey_question_columns):
                # Send RESUME LINK
                resume_url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
                message_text = f"Please complete this short follow-up survey: {resume_url}"
                status = "Partial - Resume Sent"
                count += 1
            # 3. Never started
            else:
                # Send full survey link
                full_url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
                message_text = f"Reminder to answer this short follow-up survey: {full_url}"
                status = "Not Started - Survey Link Sent"
                count += 1

            # Update the existing row in df_existing
            df_existing.loc[df_existing["Response ID"] == response_id, ["URL", "Status", "Count"]] = [message_text, status, count]
        else:
            print('Condition Not met')
    # If the response_id is not found, process as a new record
    else:
        print('not inside matching data')
        count = 0  # Start count at 0 for new records

        # 1. Survey completed?
        if any(pd.notna(row[col]) for col in survey_end_columns):
            status = "Completed"
            message_text = ""
        # 2. Partial survey?
        elif any(pd.notna(row[col]) for col in survey_question_columns):
            # Send RESUME LINK
            resume_url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
            message_text = f"Please complete this short follow-up survey: {resume_url}"
            status = "Partial - Resume Sent"
            count += 1
        # 3. Never started
        else:
            # Send full survey link
            full_url = f"https://etc-research.com/index.php/8383?lang=en&MAINSID={mainsid}&RESPONSEID={response_id}"
            message_text = f"Reminder to answer this short follow-up survey: {full_url}"
            status = "Not Started - Survey Link Sent"
            count += 1

        # Log info for new records
        data_to_save.append([phone_number, response_id, mainsid, message_text, status, count])

# Convert data_to_save to a DataFrame (only new records)
if data_to_save:
    df_to_save = pd.DataFrame(data_to_save, columns=["Phone Number", "Response ID", "Main SID", "URL", "Status", "Count"])
    
    # Combine existing data (updated in-place) with new data
    df_existing_updated = pd.concat([df_existing, df_to_save], ignore_index=True)
else:
    # If no new records, use the updated df_existing directly
    df_existing_updated = df_existing

# Write the updated data back to the CSV file
df_existing_updated.to_csv(csv_filename, index=False)

19767 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19751 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19741 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19587 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19659 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19740 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19596 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19597 <class 'int'>
932964 <class 'str'>
inside matching data
Condition Not met
19618 <class 'int'>
nan <class 'float'>
19739 <class 'int'>
nan <class 'float'>
19608 <class 'int'>
nan <class 'float'>
19752 <class 'int'>
nan <class 'float'>
19758 <class 'int'>
nan <class 'float'>
19765 <class 'int'>
nan <class 'float'>
19724 <class 'int'>
nan <class 'float'>
